# CA 1 – <Dataset name>
**Goal:** predict `<target>` from `<features>`.
Steps: 1. Method · 2. EDA · 3. Preprocessing / feature selection · 4. ML pipeline · 5. Results · 6. Ethics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

FILE = "patients.csv"       # <-- change
TARGET = "heart_disease"    # <-- change
df = pd.read_csv(FILE)
df.head()

## 1. Type of method
- Target `<target>` is **<categories / a number / not given>** → this is **<classification / regression / clustering>** (supervised / unsupervised).
- Models I will compare: **Logistic Regression** (simple, explainable baseline), **Decision Tree** (easy to explain), **Random Forest** (usually more accurate).

## 2. Exploratory Data Analysis

In [ ]:
print(df.shape)
df.info()
df.describe()

In [ ]:
print(df.isnull().sum())
print("Duplicates:", df.duplicated().sum())
print(df[TARGET].value_counts(normalize=True))

In [ ]:
df.hist(figsize=(10, 8)); plt.tight_layout(); plt.show()
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm"); plt.show()

**EDA findings**
- Rows / columns: …
- Missing values: …   Duplicates: …
- Target balance: …
- Outliers: …
- Most related to target: …

## 3. Preprocessing / Feature selection

In [ ]:
df = df.drop_duplicates()
df = df.drop(columns=["patient_id"])                                   # <-- ID columns
df = df[df["blood_pressure"] < 250]                                    # <-- impossible values
df["bmi"] = df["bmi"].fillna(df["bmi"].median())                       # <-- numeric missing
df["cholesterol"] = df["cholesterol"].fillna(df["cholesterol"].median())
df["exercise"] = df["exercise"].fillna(df["exercise"].mode()[0])       # <-- category missing
df["smoker"] = df["smoker"].map({"no": 0, "yes": 1})
df = pd.get_dummies(df, columns=["sex", "exercise"], drop_first=True, dtype=int)
print(df.isnull().sum().sum(), "missing left")
df.corr()[TARGET].sort_values(ascending=False)

**Why:** duplicates removed (would bias the model), median used because it is not affected by outliers,
one-hot encoding because models need numbers, features kept based on correlation / importance.

## 4. ML pipeline
split → scale (fit on train only) → train models → cross-validate → evaluate on test

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}
print("Baseline:", round(y_test.value_counts(normalize=True).max(), 3))
for name, model in models.items():
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    cv = cross_val_score(model, X_train_s, y_train, cv=5).mean()
    print(f"{name:20} CV={cv:.3f}  test acc={accuracy_score(y_test, pred):.3f}  f1={f1_score(y_test, pred):.3f}")

## 5. Results

In [ ]:
best = models["Logistic Regression"]      # <-- your best model
pred = best.predict(X_test_s)
print(classification_report(y_test, pred))
ConfusionMatrixDisplay.from_predictions(y_test, pred); plt.show()
pd.Series(best.coef_[0], index=X.columns).sort_values()

**Results**
- Best model: … with accuracy … and F1 … (baseline …).
- Confusion matrix: … false negatives (missed patients) – this is the costly error here.
- Most important features: …
- Limitations: small / synthetic data, only one split, …

## 6. Ethical concerns

In [ ]:
test = X_test.copy()
test["actual"], test["pred"] = y_test, pred
test.groupby("sex_M")[["actual", "pred"]].mean()      # compare rates for each group

- **Bias / fairness:** …
- **Privacy:** …
- **Transparency:** …
- **Accountability / human oversight:** …
- **Mitigation:** …